# 🚀 Custom AI Enhancer — Real-ESRGAN Super-Resolution GPU Training
- Model: RRDBNet Super-Resolution (4x Super-Resolution for Scenery, Anime & Portraits)
- Hardware: Kaggle Nvidia GPU (Tesla P100 / T4)
- Target: 5,000 Iterations + Dynamic ONNX Export

In [ ]:
import os, sys, subprocess, shutil, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    try:
        c = torch.nn.Conv2d(3, 3, 3).cuda()
        x = torch.randn(1, 3, 32, 32, device='cuda')
        _ = c(x)
        print("Native CUDA Conv2D verification PASSED!")
    except Exception as e:
        print(f"CUDA check notice ({e}). Installing compatible PyTorch 2.4.0+cu121...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
            'torch==2.4.0+cu121', 'torchvision==0.19.0+cu121',
            '--index-url', 'https://download.pytorch.org/whl/cu121'], check=False)
        print('PyTorch setup completed!')
else:
    print('Running on CPU')


In [ ]:
# 1. Clone repository & Install packages
import os, shutil
%cd /kaggle/working
if os.path.exists('custom-ai-enhancer'):
    shutil.rmtree('custom-ai-enhancer')
!git clone https://github.com/supli6669/Enhance-Image.git custom-ai-enhancer
%cd /kaggle/working/custom-ai-enhancer

!pip install -q facexlib gfpgan lpips gdown onnx 'onnxscript==0.1.0.dev20231023' onnxruntime-gpu pyyaml opencv-python scikit-image
!python tools/patch_and_install_basicsr.py

# Direct patch for torchvision functional_tensor
!python -c "import glob, os; [open(f, 'w').write(open(f).read().replace('torchvision.transforms.functional_tensor', 'torchvision.transforms.functional')) for f in glob.glob('/usr/local/lib/python*/dist-packages/basicsr/**/*.py', recursive=True) if 'functional_tensor' in open(f).read()]"

# 2. Download weights and prepare dataset
!python tools/download_weights.py
!python tools/prepare_toy_training.py
print("Environment, weights, and dataset ready!")

In [ ]:
# 3. Launch Real-ESRGAN GPU Training
%cd /kaggle/working/custom-ai-enhancer
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!python train_realesrgan.py


In [ ]:
# 4. Export Real-ESRGAN to ONNX
%cd /kaggle/working/custom-ai-enhancer
import os, sys, shutil, glob

try:
    ret = os.system("python tools/export_onnx.py")
    if ret == 0:
        print("[OK] ONNX export succeeded.")
    else:
        print("[WARN] ONNX export exited non-zero.")
except Exception as e:
    print(f"[WARN] ONNX export failed: {e}")

# Copy trained Real-ESRGAN checkpoint to /kaggle/working
pth_files = sorted(glob.glob("models/Real-ESRGAN/experiments/**/net_g_*.pth", recursive=True))
if pth_files:
    latest = pth_files[-1]
    shutil.copy(latest, "/kaggle/working/realesrgan_custom.pth")
    print(f"Checkpoint saved: {latest} -> /kaggle/working/realesrgan_custom.pth")

# Copy exported ONNX model
onnx_src = "weights/realesrgan/realesrgan.onnx"
if os.path.exists(onnx_src):
    shutil.copy(onnx_src, "/kaggle/working/realesrgan_custom.onnx")
    print(f"ONNX model saved: {onnx_src} -> /kaggle/working/realesrgan_custom.onnx")

print("\nTraining and Export finished! Outputs saved in /kaggle/working/")
print("Files available for download:")
for f in os.listdir("/kaggle/working"):
    fp = f"/kaggle/working/{f}"
    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / 1024 / 1024
        print(f"  {f}: {size_mb:.1f} MB")
